# Tarea 13 - Agente RAG con LangChain

### Tema principal: Energías renovables
### Tema oculto: Aeronaves comerciales

___

___

## Dependencias

In [3]:
from pathlib import Path

import json
import re

from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool

___

___

## Configuración

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Paths
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ROOT = Path.cwd()
DOCS_DIR = ROOT / "docs"
CHROMA_DIR = ROOT / "vectors_langchain"
COLLECTION_NAME = "nlp_docs_langchain"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Modelos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OLLAMA_MODEL = "granite4.1:3b"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Parámetros
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CHUNK_SIZE = 700
CHUNK_OVERLAP = 150
TOP_K = 3
WEB_MAX_RESULTS = 5

DOCS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

___

___

## Modelos

In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LLM
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Embeddings
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14139.34it/s]


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Prueba rápida del modelo
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
response = llm.invoke(
    "Explain briefly what a RAG system is."
)

print(response.content)

A RAG (Retrieval-Augmented Generation) system is an AI architecture that combines traditional machine learning models with information retrieval techniques to enhance the generation of text or responses. Here’s a brief explanation:

1. **Information Retrieval**: The system first retrieves relevant documents, passages, or data from a large corpus (like a database or a collection of web pages) based on the input query or context.

2. **Contextual Understanding**: These retrieved pieces of information provide context to the AI model, allowing it to generate more accurate, informed, and coherent responses tailored to the specific question or task at hand.

3. **Generation**: Using the contextual data gathered, the model then generates its output—whether that’s answering a query, summarizing text, translating content, or creating new text based on patterns learned from the retrieved information.

The key benefit of RAG systems is their ability to leverage up-to-date and domain-specific know

___

___

## Sistema RAG

### Documentos

In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Carga de documentos PDF
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_pdf_documents(docs_dir: Path) -> list:
    documents = []

    pdf_files = sorted(docs_dir.glob("*.pdf"))

    for pdf_path in pdf_files:
        loader = PyMuPDFLoader(
            file_path=str(pdf_path)
        )

        loaded_documents = loader.load()

        for document in loaded_documents:
            document.metadata["source"] = pdf_path.name

        documents.extend(loaded_documents)

    return documents


documents = load_pdf_documents(DOCS_DIR)

print("Documentos cargados:", len(documents))

for document in documents[:3]:
    print("SOURCE:", document.metadata.get("source"))
    print("PAGE:", document.metadata.get("page"))
    print(document.page_content[:500])
    print("-" * 80)

Documentos cargados: 372
SOURCE: 27955.pdf
PAGE: 0
What is Renewable Energy?
Renewable energy uses energy sources
that are continually replenished by
nature—the sun, the wind, water, the
Earth’s heat, and plants. Renewable
energy technologies turn these fuels into
usable forms of energy—most often elec-
tricity, but also heat, chemicals, or
mechanical power.
Why Use Renewable Energy?
Today we primarily use fossil fuels to heat
and power our homes and fuel our cars.
It’s convenient to use coal, oil, and natural
gas for meeting our energy needs, b
--------------------------------------------------------------------------------
SOURCE: 27955.pdf
PAGE: 1
Renewable energy will also help us
develop energy independence and secu-
rity. The United States imports more than
50 percent of its oil, up from 34 percent in
1973. Replacing some of our petroleum
with fuels made from plant matter, for
example, could save money and
strengthen our energy security.
Renewable energy is plentiful, and the
tec

### Chunks

In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# División de documentos en chunks
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = index

print("Chunks creados:", len(chunks))

for chunk in chunks[:3]:
    print("SOURCE:", chunk.metadata.get("source"))
    print("PAGE:", chunk.metadata.get("page"))
    print("CHUNK:", chunk.metadata.get("chunk_index"))
    print(chunk.page_content[:500])
    print("-" * 80)

Chunks creados: 2516
SOURCE: 27955.pdf
PAGE: 0
CHUNK: 0
What is Renewable Energy?
Renewable energy uses energy sources
that are continually replenished by
nature—the sun, the wind, water, the
Earth’s heat, and plants. Renewable
energy technologies turn these fuels into
usable forms of energy—most often elec-
tricity, but also heat, chemicals, or
mechanical power.
Why Use Renewable Energy?
Today we primarily use fossil fuels to heat
and power our homes and fuel our cars.
It’s convenient to use coal, oil, and natural
gas for meeting our energy needs, b
--------------------------------------------------------------------------------
SOURCE: 27955.pdf
PAGE: 0
CHUNK: 1
Earth. We’re using them much more
rapidly than they are being created. Even-
tually, they will run out. And because of 
safety concerns and waste disposal prob-
lems, the United States will retire much of
its nuclear capacity by 2020. In the mean-
time, the nation’s energy needs are
expected to grow by 33 percent during the
n

### ChromaDB

In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Base vectorial con Chroma
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR)
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Indexación de documentos
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if vector_store._collection.count() == 0 and chunks:
    vector_store.add_documents(
        documents=chunks
    )

print("Chunks indexados en Chroma:", vector_store._collection.count())

Chunks indexados en Chroma: 2516


In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Búsqueda de prueba
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
test_results = vector_store.similarity_search_with_score(
    query="What is renewable energy?",
    k=TOP_K
)

for document, score in test_results:
    print("SOURCE:", document.metadata.get("source"))
    print("PAGE:", document.metadata.get("page"))
    print("SCORE:", score)
    print(document.page_content[:500])
    print("-" * 80)

SOURCE: 27955.pdf
PAGE: 0
SCORE: 0.445686936378479
What is Renewable Energy?
Renewable energy uses energy sources
that are continually replenished by
nature—the sun, the wind, water, the
Earth’s heat, and plants. Renewable
energy technologies turn these fuels into
usable forms of energy—most often elec-
tricity, but also heat, chemicals, or
mechanical power.
Why Use Renewable Energy?
Today we primarily use fossil fuels to heat
and power our homes and fuel our cars.
It’s convenient to use coal, oil, and natural
gas for meeting our energy needs, b
--------------------------------------------------------------------------------
SOURCE: SFJD+070.pdf
PAGE: 1
SCORE: 0.48729681968688965
(Abdullah 2016, 160). 
 
2.2 DEFINITION OF RENEWABLE ENERGY 
 
Renewable energy refers to energy that recurs naturally and periodically in nature. It is energy 
generated from an inexhaustible natural source, readily available across the Earth's surface, and can be 
easily converted into usable energy. Renewab

### RAG Tool

In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Construcción de contexto
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def build_rag_context(used_chunks: list[dict]) -> str:
    context_parts = []

    for chunk in used_chunks:
        context_parts.append(
            f"""Fuente: {chunk["source"]}, página {chunk["page"]}
Chunk: {chunk["chunk_index"]}
Contenido:
{chunk["text"]}"""
        )

    return "\n\n".join(context_parts)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG Tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@tool
def rag_search_tool(query: str, top_k: int = TOP_K) -> dict:
    """
    Busca información relevante en los documentos PDF locales usando Chroma.

    Args:
        query: Pregunta o consulta del usuario.
        top_k: Número de fragmentos relevantes a recuperar.
    """

    results = vector_store.similarity_search_with_score(
        query=query,
        k=top_k
    )

    if not results:
        return {
            "tool_name": "rag_search_tool",
            "found": False,
            "context": "",
            "sources": [],
            "used_chunks": [],
            "best_score": None,
            "reason": "No se recuperaron chunks desde Chroma."
        }

    sources = []
    used_chunks = []

    for document, score in results:
        raw_page = document.metadata.get("page")
        page = raw_page + 1 if isinstance(raw_page, int) else raw_page

        source = document.metadata.get("source", "unknown")
        chunk_index = document.metadata.get("chunk_index", "unknown")

        sources.append(
            {
                "source": source,
                "page": page,
                "chunk_index": chunk_index,
                "score": score
            }
        )

        used_chunks.append(
            {
                "source": source,
                "page": page,
                "chunk_index": chunk_index,
                "score": score,
                "text": document.page_content
            }
        )

    context = build_rag_context(
        used_chunks=used_chunks
    )

    best_score = min(
        chunk["score"] for chunk in used_chunks
    )

    return {
        "tool_name": "rag_search_tool",
        "found": True,
        "context": context,
        "sources": sources,
        "used_chunks": used_chunks,
        "best_score": best_score,
        "reason": "Se recuperaron chunks desde los PDFs locales."
    }

In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Prueba de RAG Tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
rag_result = rag_search_tool.invoke(
    {
        "query": "What is renewable energy?",
        "top_k": TOP_K
    }
)

print("TOOL:", rag_result["tool_name"])
print("FOUND:", rag_result["found"])
print("BEST SCORE:", rag_result["best_score"])
print("SOURCES:", rag_result["sources"])
print()
print("CONTEXT:")
print(rag_result["context"][:1500])

TOOL: rag_search_tool
FOUND: True
BEST SCORE: 0.445686936378479
SOURCES: [{'source': '27955.pdf', 'page': 1, 'chunk_index': 0, 'score': 0.445686936378479}, {'source': 'SFJD+070.pdf', 'page': 2, 'chunk_index': 438, 'score': 0.48729681968688965}, {'source': 'RenewableEnergy_FastFacts.pdf', 'page': 1, 'chunk_index': 423, 'score': 0.5766503214836121}]

CONTEXT:
Fuente: 27955.pdf, página 1
Chunk: 0
Contenido:
What is Renewable Energy?
Renewable energy uses energy sources
that are continually replenished by
nature—the sun, the wind, water, the
Earth’s heat, and plants. Renewable
energy technologies turn these fuels into
usable forms of energy—most often elec-
tricity, but also heat, chemicals, or
mechanical power.
Why Use Renewable Energy?
Today we primarily use fossil fuels to heat
and power our homes and fuel our cars.
It’s convenient to use coal, oil, and natural
gas for meeting our energy needs, but we
have a limited supply of these fuels on the
Earth. We’re using them much more
rapidly 

___

___

## Web Search

In [13]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Web Search Tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
web_search_runner = DuckDuckGoSearchRun()


@tool
def web_search_tool(query: str, max_results: int = WEB_MAX_RESULTS) -> dict:
    """
    Busca información en internet usando DuckDuckGoSearchRun.

    Args:
        query: Pregunta o consulta del usuario.
        max_results: Número máximo de resultados a considerar.
    """

    try:
        search_result = web_search_runner.invoke(query)

        if not search_result or not search_result.strip():
            return {
                "tool_name": "web_search_tool",
                "found": False,
                "context": "",
                "sources": ["DuckDuckGo"],
                "reason": "DuckDuckGo no devolvió resultados útiles."
            }

        return {
            "tool_name": "web_search_tool",
            "found": True,
            "context": search_result,
            "sources": ["DuckDuckGo"],
            "reason": "Se encontró información en internet usando DuckDuckGoSearchRun."
        }

    except Exception as error:
        return {
            "tool_name": "web_search_tool",
            "found": False,
            "context": "",
            "sources": ["DuckDuckGo"],
            "reason": f"Ocurrió un error al buscar en internet: {error}"
        }

In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Prueba de Web Search Tool
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
web_result = web_search_tool.invoke(
    {
        "query": "What is renewable energy?",
        "max_results": WEB_MAX_RESULTS
    }
)

print("TOOL:", web_result["tool_name"])
print("FOUND:", web_result["found"])
print("SOURCES:", web_result["sources"])
print("REASON:", web_result["reason"])
print()
print("CONTEXT:")
print(web_result["context"][:1500])

TOOL: web_search_tool
FOUND: True
SOURCES: ['DuckDuckGo']
REASON: Se encontró información en internet usando DuckDuckGoSearchRun.

CONTEXT:
1 week ago - Renewable energy (also called green energy) is energy made from renewable natural resources that are replenished on a human timescale. The most widely used renewable energy types are solar energy, wind power, and hydropower. Bioenergy and geothermal power are also significant in some countries. April 13, 2026 - Renewable energy is energy generated from natural resources—such as sunlight, wind, rain, tides and geothermal heat. March 20, 2026 - Renewable energy is energy generated from natural sources that are replenished faster than they are used. October 22, 2025 - According to the National Renewable Energy Laboratory, “more energy from the sun falls on the earth in one hour than is used by everyone in the world in one year.” Today, we use the sun’s rays in many ways—to heat homes and businesses, to warm water, and to power devices. Ma

___

___

## Evaluador de contexto

In [15]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Utilidad para limpiar JSON
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def clean_json_response(raw_response: str) -> str:
    cleaned_response = raw_response.strip()

    cleaned_response = re.sub(
        pattern=r"^```json\s*",
        repl="",
        string=cleaned_response
    )

    cleaned_response = re.sub(
        pattern=r"^```\s*",
        repl="",
        string=cleaned_response
    )

    cleaned_response = re.sub(
        pattern=r"\s*```$",
        repl="",
        string=cleaned_response
    )

    return cleaned_response.strip()

In [16]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Evaluador de contexto
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
rag_evaluator_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Eres un evaluador de contexto para un sistema RAG.

Tu tarea es decidir si el contexto proporcionado contiene información suficiente
para responder la pregunta del usuario.

Reglas:
- No respondas la pregunta del usuario.
- Solo evalúa si el contexto es suficiente.
- Si el contexto contiene una respuesta clara a la pregunta, responde true.
- Si el contexto habla de otro tema, responde false.
- Si el contexto solo menciona palabras relacionadas, pero no responde realmente, responde false.
- Responde únicamente en formato JSON válido.

El JSON debe tener exactamente esta estructura:
{{
    "is_enough": true,
    "reason": "explicación breve"
}}
"""
        ),
        (
            "user",
            """
Pregunta del usuario:
{query}

Contexto recuperado desde el RAG:
{context}
"""
        )
    ]
)

rag_evaluator_chain = rag_evaluator_prompt | llm | StrOutputParser()


@tool
def evaluate_rag_context(query: str, context: str) -> dict:
    """
    Evalúa si el contexto recuperado por RAG es suficiente para responder la pregunta.

    Args:
        query: Pregunta original del usuario.
        context: Contexto recuperado desde los documentos PDF locales.
    """

    if not context.strip():
        return {
            "tool_name": "evaluate_rag_context",
            "is_enough": False,
            "reason": "El contexto recuperado desde el RAG está vacío."
        }

    raw_response = rag_evaluator_chain.invoke(
        {
            "query": query,
            "context": context
        }
    )

    cleaned_response = clean_json_response(raw_response)

    try:
        evaluation = json.loads(cleaned_response)

        return {
            "tool_name": "evaluate_rag_context",
            "is_enough": bool(evaluation.get("is_enough", False)),
            "reason": evaluation.get("reason", "No se proporcionó una razón.")
        }

    except json.JSONDecodeError:
        return {
            "tool_name": "evaluate_rag_context",
            "is_enough": False,
            "reason": f"El evaluador no devolvió JSON válido: {raw_response}"
        }

In [17]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Prueba de evaluador
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
evaluation = evaluate_rag_context.invoke(
    {
        "query": "What is renewable energy?",
        "context": rag_result["context"]
    }
)

print(evaluation)

{'tool_name': 'evaluate_rag_context', 'is_enough': True, 'reason': 'El contexto proporciona una definición clara de la energía renovable, explicando que proviene de fuentes naturales y continuamente reponibles como el sol, viento y agua. También menciona sus ventajas ambientales y contrasta con los combustibles fósiles, satisfaciendo completamente la pregunta sobre qué es la energía renovable.'}


___

___

## Agent

### System prompt

In [18]:
AGENT_SYSTEM_PROMPT = """
Eres un agente de preguntas y respuestas especializado en Procesamiento de Lenguaje Natural.

Tu objetivo es responder preguntas del usuario usando la mejor fuente disponible.

Reglas principales:
1. Siempre debes intentar usar primero la información recuperada desde el sistema RAG local.
2. Si el sistema RAG contiene información suficiente, responde usando únicamente ese contexto.
3. Si el sistema RAG no contiene información suficiente, utiliza información recuperada desde internet.
4. No inventes información.
5. Si ninguna fuente contiene información suficiente, dilo claramente.
6. Cuando uses información del RAG, menciona las fuentes PDF y páginas si están disponibles.
7. Cuando uses información de internet, menciona que la información proviene de búsqueda web.
8. Responde de forma clara, ordenada y útil para un estudiante de Procesamiento de Lenguaje Natural.

No debes explicar el funcionamiento interno del agente a menos que el usuario lo pregunte.
"""

### Tool map

In [19]:
TOOLS = [
    rag_search_tool,
    evaluate_rag_context,
    web_search_tool
]

TOOL_MAP = {
    current_tool.name: current_tool
    for current_tool in TOOLS
}

for tool_name, current_tool in TOOL_MAP.items():
    print(tool_name, "→", current_tool.description)

rag_search_tool → Busca información relevante en los documentos PDF locales usando Chroma.

Args:
    query: Pregunta o consulta del usuario.
    top_k: Número de fragmentos relevantes a recuperar.
evaluate_rag_context → Evalúa si el contexto recuperado por RAG es suficiente para responder la pregunta.

Args:
    query: Pregunta original del usuario.
    context: Contexto recuperado desde los documentos PDF locales.
web_search_tool → Busca información en internet usando DuckDuckGoSearchRun.

Args:
    query: Pregunta o consulta del usuario.
    max_results: Número máximo de resultados a considerar.


### Generador de respuesta

In [20]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Generador de respuesta
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            AGENT_SYSTEM_PROMPT
        ),
        (
            "user",
            """
Pregunta del usuario:
{query}

Tipo de fuente:
{source_type}

Instrucciones sobre la fuente:
{source_instruction}

Contexto disponible:
{context}

Respuesta:
"""
        )
    ]
)

answer_chain = answer_prompt | llm | StrOutputParser()


def generate_answer_with_context(query: str, context: str, source_type: str) -> str:
    if source_type == "rag":
        source_instruction = """
La información proviene del sistema RAG local.
Usa únicamente el contexto proporcionado.
Menciona las fuentes PDF y páginas cuando estén disponibles.
"""

    elif source_type == "web":
        source_instruction = """
La información proviene de una búsqueda web.
Usa únicamente el contexto proporcionado.
Indica que la información fue obtenida mediante búsqueda web.
"""

    else:
        source_instruction = """
Usa únicamente el contexto proporcionado.
"""

    answer = answer_chain.invoke(
        {
            "query": query,
            "context": context,
            "source_type": source_type,
            "source_instruction": source_instruction
        }
    )

    return answer

### Formato para fuentes

In [21]:
def format_rag_sources(used_chunks: list[dict]) -> str:
    if not used_chunks:
        return "No se encontraron fuentes RAG."

    formatted_sources = []

    for index, chunk in enumerate(used_chunks, start=1):
        formatted_sources.append(
            f"""
Fuente {index}
PDF: {chunk["source"]}
Página: {chunk["page"]}
Chunk: {chunk["chunk_index"]}
Score: {chunk["score"]}

Texto utilizado:
{chunk["text"]}
"""
        )

    return "\n".join(formatted_sources)

### Loop

In [22]:
def agent_loop(user_query: str, verbose: bool = True) -> dict:
    if verbose:
        print("Agent: buscando información en el RAG local...")

    rag_result = TOOL_MAP["rag_search_tool"].invoke(
        {
            "query": user_query,
            "top_k": TOP_K
        }
    )

    if verbose:
        print("Agent: evaluando si el contexto del RAG es suficiente...")

    rag_evaluation = TOOL_MAP["evaluate_rag_context"].invoke(
        {
            "query": user_query,
            "context": rag_result["context"]
        }
    )

    if rag_evaluation["is_enough"]:
        if verbose:
            print("Agent: el RAG tiene información suficiente.")
            print("Agent: generando respuesta final con contexto RAG...")

        answer = generate_answer_with_context(
            query=user_query,
            context=rag_result["context"],
            source_type="rag"
        )

        rag_sources_text = format_rag_sources(
            rag_result.get("used_chunks", [])
        )

        return {
            "answer": answer,
            "source_type": "rag",
            "rag_result": rag_result,
            "rag_evaluation": rag_evaluation,
            "rag_sources_text": rag_sources_text,
            "web_result": None
        }

    if verbose:
        print("Agent: el RAG no tiene información suficiente.")
        print("Agent: buscando información en internet...")

    web_result = TOOL_MAP["web_search_tool"].invoke(
        {
            "query": user_query,
            "max_results": WEB_MAX_RESULTS
        }
    )

    if not web_result["found"]:
        return {
            "answer": (
                "No encontré información suficiente en los documentos locales "
                "ni en la búsqueda web para responder con confianza."
            ),
            "source_type": "none",
            "rag_result": rag_result,
            "rag_evaluation": rag_evaluation,
            "rag_sources_text": None,
            "web_result": web_result
        }

    if verbose:
        print("Agent: generando respuesta final con contexto web...")

    answer = generate_answer_with_context(
        query=user_query,
        context=web_result["context"],
        source_type="web"
    )

    return {
        "answer": answer,
        "source_type": "web",
        "rag_result": rag_result,
        "rag_evaluation": rag_evaluation,
        "rag_sources_text": None,
        "web_result": web_result
    }

### Interfaz de respuesta

In [23]:
def ask_agent(question: str, show_sources: bool = True) -> None:
    result = agent_loop(
        user_query=question,
        verbose=True
    )

    print()
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print("Respuesta final")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(result["answer"])
    print()
    print("Fuente utilizada:", result["source_type"])

    if show_sources and result["source_type"] == "rag":
        print()
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print("Fuentes RAG utilizadas")
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(result["rag_sources_text"])

___

___

## Testeo de querys

In [24]:
test_queries = [
    {
        "query": "What is renewable energy and what are its main sources?",
        "expected_case": "in_domain_good_rag_result"
    },
    {
        "query": "How does renewable energy help reduce greenhouse gas emissions?",
        "expected_case": "in_domain_good_rag_result"
    },
    {
        "query": "What are the main challenges for renewable energy deployment in the European Union?",
        "expected_case": "in_domain_good_rag_result"
    },
    {
        "query": "What is the best renewable energy source for a small restaurant in Hermosillo?",
        "expected_case": "in_domain_poor_rag_result"
    },
    {
        "query": "Who won the FIFA World Cup in 2022?",
        "expected_case": "out_of_domain"
    }
]

In [25]:
for test in test_queries:
    print("QUERY:", test["query"])
    print("EXPECTED:", test["expected_case"])
    print()

    result = agent_loop(
        user_query=test["query"],
        verbose=True
    )

    print()
    print("SOURCE USED:", result["source_type"])
    print("RAG EVALUATION:", result["rag_evaluation"])
    print("ANSWER:")
    print(result["answer"])
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

QUERY: What is renewable energy and what are its main sources?
EXPECTED: in_domain_good_rag_result

Agent: buscando información en el RAG local...
Agent: evaluando si el contexto del RAG es suficiente...
Agent: el RAG no tiene información suficiente.
Agent: buscando información en internet...
Agent: generando respuesta final con contexto web...

SOURCE USED: web
RAG EVALUATION: {'tool_name': 'evaluate_rag_context', 'is_enough': False, 'reason': 'El contexto proporcionado explica qué es la energía renovable y sus beneficios, pero no lista o describe claramente las principales fuentes de energía renovable como el sol, viento, agua, etc.'}
ANSWER:
Renewable energy, también conocida como energía verde, es la energía generada a partir de recursos naturales renovables que se reponen a una escala humana. Las principales fuentes de energía renovable incluyen la energía solar, la energía eólica y la hidroenergía. Otras fuentes significativas son la bioenergia y la energía geotérmica, que son im

In [26]:
ask_agent("What is renewable energy?")

Agent: buscando información en el RAG local...
Agent: evaluando si el contexto del RAG es suficiente...
Agent: el RAG tiene información suficiente.
Agent: generando respuesta final con contexto RAG...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Respuesta final
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Renewable energy se refiere a energía que se recibe natural e incesantemente de la naturaleza. Se genera a partir de fuentes naturales inexhaustibles y ampliamente disponibles en la superficie de la Tierra, y puede convertirse fácilmente en energía utilizable. Las características principales del energía renovable incluyen ser amigable con el medio ambiente, limpia y no contaminante. A diferencia de las fuentes de energía no renovables, que se agotan, el uso de energía renovable no lleva a su agotamiento.

Las fuentes disponibles en el sistema RAG local son:

1. Fuente: 27955.pdf, página 1  
   Contenido: "Renewable energy uses energy sources that are c